# BitBully — Bitboard Internals Visualizer

This notebook demonstrates how the [BitBully](https://github.com/MarkusThill/BitBully) Connect-4 engine represents the game board internally using **bitboards** — 64-bit integers where individual bits map to board cells.

## Key Concepts Illustrated

1. **Bit Layout** — Each column occupies 9 bits (6 playable + 3 sentinel) in a `uint64_t`
2. **Two-Bitboard Representation** — `allTokens` (all pieces) + `activePTokens` (current player), with XOR to derive the opponent
3. **Legal Move Generation** — The carry-propagation trick: `(allTokens + BB_BOTTOM_ROW) & BB_ALL_LEGAL`
4. **Win Detection** — Shift-AND technique checking 4 directions in parallel
5. **Winning Positions / Threats** — Where placing a token would complete four-in-a-row

In [ ]:
%matplotlib inline
from techdays26.gui_bitboard import BitboardVisualizer

## Interactive Visualizer

Use the **column buttons** (⏬0–⏬6) or the **Moves** text field to build a position. Select different **overlays** to explore:

| Overlay | What it shows |
|---|---|
| **None** | Opponent's tokens (derived via XOR) |
| **Legal Moves** | Carry-propagation trick in action |
| **Winning Positions** | Threat cells for the current player |
| **Win Check: ...** | Step-by-step shift-AND win detection |
| **Bit Index Map** | Which bit index maps to which cell |

In [ ]:
# Launch with an empty board — click columns to place pieces
%matplotlib ipympl
vis = BitboardVisualizer()
vis.show()

## Pre-loaded Example: Near-Win Position

The position below (`3334445`) has Yellow threatening a vertical win in column 3. Select the **"Win Check: Vertical"** or **"Winning Positions"** overlay to see the threat detection in action.

In [ ]:
%matplotlib ipympl

# Pre-loaded position with a vertical threat
vis2 = BitboardVisualizer(init_moves="3334445")
vis2.show()

## Pre-loaded Example: Horizontal Win

Position `03142536` — Red completes a horizontal four-in-a-row at the bottom (columns 3–6). Select **"Win Check: Horizontal"** to see the shift-AND detection light up.

In [ ]:
# Horizontal win by Red (player 2)
vis3 = BitboardVisualizer(init_moves="03142536")
vis3.show()

## How the Bitboard Layout Works

```
Bit indices in a uint64_t (9 bits per column, 7 columns = 63 bits used):

         C0    C1    C2    C3    C4    C5    C6
       ┌─────┬─────┬─────┬─────┬─────┬─────┬─────┐
  S2   │  8  │ 17  │ 26  │ 35  │ 44  │ 53  │ 62  │  ← sentinel (illegal)
  S1   │  7  │ 16  │ 25  │ 34  │ 43  │ 52  │ 61  │  ← sentinel (illegal)
  S0   │  6  │ 15  │ 24  │ 33  │ 42  │ 51  │ 60  │  ← sentinel (illegal)
       ├─────┼─────┼─────┼─────┼─────┼─────┼─────┤
  R5   │  5  │ 14  │ 23  │ 32  │ 41  │ 50  │ 59  │  ← top playable row
  R4   │  4  │ 13  │ 22  │ 31  │ 40  │ 49  │ 58  │
  R3   │  3  │ 12  │ 21  │ 30  │ 39  │ 48  │ 57  │
  R2   │  2  │ 11  │ 20  │ 29  │ 38  │ 47  │ 56  │
  R1   │  1  │ 10  │ 19  │ 28  │ 37  │ 46  │ 55  │
  R0   │  0  │  9  │ 18  │ 27  │ 36  │ 45  │ 54  │  ← bottom row
       └─────┴─────┴─────┴─────┴─────┴─────┴─────┘
```

### Why 9 bits per column?
The 3 sentinel bits per column are crucial for the **legal move generation trick**: adding `BB_BOTTOM_ROW` to `allTokens` causes carry bits to propagate upward within each column. The sentinel bits act as a "buffer zone" that absorbs carry from full columns, preventing it from leaking into the next column.

### The two bitboards
- **`allTokens`** — every piece on the board (both players)
- **`activePTokens`** — only the *current* player's pieces

The opponent's pieces are never stored explicitly — they're computed on-the-fly as `activePTokens ⊕ allTokens` (XOR). After each move, the player swap is done with a single XOR: `activePTokens ^= allTokens`.